In [0]:
%run ../init_framework_silver

In [0]:
# 1. SOURCE INGESTION
# Read from Bronze CDF.
df_bronze_defaulters = read_delta_stream(
    spark=spark, 
    table_name=DEFAULTERS_BRONZE
)

In [0]:
# 3. METADATA ENRICHMENT
# Injects standard audit columns (_ingested_at, etc.)
df_with_metadata = apply_silver_metadata(df_bronze_defaulters)

In [0]:
# 4. DELINQUENCY FIX
from pyspark.sql.functions import col

# Fixing the delinquency column to be an integer 
    # -- Cast to float first to avoid losing decimal strings to NULL
    # -- Cast to int to perform the "floor" truncation
    # -- Replace all NULLs (original nulls + invalid strings) with 0

df_delinq2yrs_fixed = df_with_metadata.withColumn(
    "delinq_2yrs", col("delinq_2yrs").cast("float").cast("int")
).fillna(0, subset=["delinq_2yrs"]) 

In [0]:
# 5. DELINQUENCY ENTITY
from pyspark.sql.functions import col

# filter logic: only keep records where there's actually a delinquency event to track.
df_defaulters_delinq = (df_delinq2yrs_fixed
    .select(
        "member_id",
        "delinq_2yrs",
        "delinq_amnt",
        col("mths_since_last_delinq").cast("int"),
        # Metadata Columns
        "_ingested_at",
        "_source_file",
        "_bronze_version",
        "_updated_at",
        "_change_type",
        "_batch_id"
    )
    .filter((col("delinq_2yrs") > 0) | (col("mths_since_last_delinq") > 0))
)

In [0]:
# 6. SINK: DELINQUENCY TABLE

def upsert_defaulters_delinq_to_silver(microBatchDF, batchId):
    # standard merge with version fencing to protect against out-of-order data.
    spark_session = microBatchDF.sparkSession

    from pyspark.sql import Window
    from pyspark.sql.functions import col, row_number, desc
        
    # Filter for valid CDC events (insert or post-image)
    df_updated = microBatchDF.filter(col("_change_type").isin("insert", "update_postimage"))

    # 1. Define the Window
    # Partition by the Primary Key and order by timestamp descending.
    # This ensures the freshest CDC event always receives rank 1.
    window = Window.partitionBy("member_id").orderBy(col("_updated_at").desc())

    # 2. Add the rank column ONCE
    # This creates a single "Parent" plan for both branches
    df_ranked = df_updated.withColumn("rn", row_number().over(window))

    # --- Branch Logic ---
    # 3. Clean: The absolute latest truth (rn=1) for the Silver Layer MERGE
    df_clean = df_ranked.filter(col("rn") == 1).drop("rn")

    # 4. Bad/Stale: Older intermediate updates (rn>1) for the Audit/Rejected table
    df_bad = df_ranked.filter(col("rn") > 1).drop("rn")

    # --- EXECUTE YOUR WRITES HERE ---    
    # 5. Register Clean Source View for SQL transformations
    df_clean.createOrReplaceTempView("defaulters_delinq_batch_updates")
    
    merge_query = f"""
    MERGE INTO {DEFAULTERS_DELINQ_SILVER} AS target
        USING defaulters_delinq_batch_updates AS source
        ON target.member_id = source.member_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.delinq_2yrs = source.delinq_2yrs,
            target.delinq_amnt = source.delinq_amnt,
            target.mths_since_last_delinq = source.mths_since_last_delinq,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at,
            target._batch_id = source._batch_id
        WHEN NOT MATCHED THEN
          INSERT (
            member_id, delinq_2yrs, delinq_amnt, mths_since_last_delinq,
            _ingested_at, _source_file, _bronze_version, _updated_at, _batch_id
          )
          VALUES (
            source.member_id, source.delinq_2yrs, source.delinq_amnt, source.mths_since_last_delinq,
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at, _batch_id
          )
    """
    spark_session.sql(merge_query)

# tuning for local 8-core cluster
spark.conf.set("spark.sql.shuffle.partitions", "16")

# Execute trigger-once pipeline for batch-like efficiency on streaming logic.
query_defaulters_delinq = (
    df_defaulters_delinq.writeStream
    .outputMode("append") 
    .option("checkpointLocation", SILVER_CHECKPOINT_DEFAULTERS_DELINQ)
    .trigger(availableNow=True)
    .foreachBatch(upsert_defaulters_delinq_to_silver)
    .start()
)

# block notebook termination until micro-batch is committed
query_defaulters_delinq.awaitTermination()

In [0]:
spark.sql(f"select count(1) from {DEFAULTERS_DELINQ_SILVER};")

In [0]:
# 7. ENQUIRY ENTITY
from pyspark.sql.functions import col

# pull inquiry fields for records with any history of public records or inquiries.
df_defaulters_inquiry = (
    df_delinq2yrs_fixed
    # 1. Cast immediately to align with the Silver DDL
    .withColumn("pub_rec", col("pub_rec").cast("int"))
    .withColumn("pub_rec_bankruptcies", col("pub_rec_bankruptcies").cast("int"))
    .withColumn("inq_last_6mths", col("inq_last_6mths").cast("int"))
    # 2. Select the defined schema
    .select(
        "member_id",
        "pub_rec",
        "pub_rec_bankruptcies",
        "inq_last_6mths",
        "_ingested_at",
        "_source_file",
        "_bronze_version",
        "_updated_at",
        "_change_type",
        "_batch_id"
    )
    # 3. Filter using Integer literals (0 instead of 0.0)
    .filter(
        (col("pub_rec") > 0)
        | (col("pub_rec_bankruptcies") > 0)
        | (col("inq_last_6mths") > 0)
    )
)

In [0]:
# 8. SINK: ENQUIRIES
def upsert_defaulters_inquiry_to_silver(microBatchDF, batchId):
    
    spark_session = microBatchDF.sparkSession
    
    from pyspark.sql import Window
    from pyspark.sql.functions import col, row_number, desc
        
    # Filter for valid CDC events (insert or post-image)
    df_updated = microBatchDF.filter(col("_change_type").isin("insert", "update_postimage"))

    # 1. Define the Window
    # Partition by the Primary Key and order by timestamp descending.
    # This ensures the freshest CDC event always receives rank 1.
    window = Window.partitionBy("member_id").orderBy(col("_updated_at").desc())

    # 2. Add the rank column ONCE
    # This creates a single "Parent" plan for both branches
    df_ranked = df_updated.withColumn("rn", row_number().over(window))

    # --- Branch Logic ---
    # 3. Clean: The absolute latest truth (rn=1) for the Silver Layer MERGE
    df_clean = df_ranked.filter(col("rn") == 1).drop("rn")

    # 4. Bad/Stale: Older intermediate updates (rn>1) for the Audit/Rejected table
    df_bad = df_ranked.filter(col("rn") > 1).drop("rn")

    # --- EXECUTE YOUR WRITES HERE ---    
    # 5. Register Clean Source View for SQL transformations
    df_clean.createOrReplaceTempView("defaulters_inquiry_batch_updates")

    merge_query = f"""
    MERGE INTO {DEFAULTERS_INQUIRY_SILVER} AS target
        USING defaulters_inquiry_batch_updates AS source
        ON target.member_id = source.member_id
        WHEN MATCHED AND source._bronze_version > target._bronze_version THEN
          UPDATE SET
            target.pub_rec = source.pub_rec,
            target.pub_rec_bankruptcies = source.pub_rec_bankruptcies,
            target.inq_last_6mths = source.inq_last_6mths,
            target._ingested_at = source._ingested_at,
            target._source_file = source._source_file,
            target._bronze_version = source._bronze_version,
            target._updated_at = source._updated_at,
            target._batch_id = source._batch_id
        WHEN NOT MATCHED THEN
          INSERT (
            member_id, pub_rec, pub_rec_bankruptcies, inq_last_6mths,
            _ingested_at, _source_file, _bronze_version, _updated_at, _batch_id
          )
          VALUES (
            source.member_id, source.pub_rec, source.pub_rec_bankruptcies, source.inq_last_6mths,
            source._ingested_at, source._source_file, source._bronze_version, source._updated_at, _batch_id
          )
    """
    spark_session.sql(merge_query)

query_defaulters_inquiry = (
    df_defaulters_inquiry.writeStream
    .outputMode("append") 
    .option("checkpointLocation", SILVER_CHECKPOINT_DEFAULTERS_INQUIRY)
    .trigger(availableNow=True)
    .foreachBatch(upsert_defaulters_inquiry_to_silver)
    .start()
)

query_defaulters_inquiry.awaitTermination()

In [0]:
spark.sql(f"select count(1) from {DEFAULTERS_INQUIRY_SILVER};")